# Prerequisites

## Import Libs

In [1]:

# Import libraries
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, AutoMinorLocator
import seaborn as sns

import joblib
import pickle

sns.set_style('darkgrid')

from catboost import CatBoostClassifier

from sklearn.preprocessing import PolynomialFeatures

from pathlib import Path
# research_path = Path('../') # 'utils_functionality/')

import os
import sys
sys.path.append('../')

import utils_functionality.model_analysis as ma

,impact_type,value,color,marker,order
0,no splashing,0,b,x,0
1,splashing,2,r,o,2
2,semi splashing,1,g,s,1


,impact_type,value,color,marker,order
0,fragmentation,0,r,x,0
1,bulk deformation,1,b,o,1


## Dataframes preparation

In [2]:
# import new main notebook and add splashing_spectrum for plotting
df = pd.read_excel('../data/df_modelling_no_multicollinearity.xlsx')
df_main = pd.read_excel('../data/df_main.xlsx')
df['splashing_spectrum'] = df_main['splashing_spectrum']

df = ma.get_text_definitions(df)

df['volume_fraction_type'] = (
    df['volume_fraction_binary'].apply(ma.get_volume_fraction_type)
)

display(df.head())
df.info()

,net_impact,splashing,wettability,liquid_density,surface_tension,viscosity,droplet_diameter,inclination,roughness_binary,particle_liquid_density_ratio,volume_fraction_binary,particle_droplet_diameter_ratio,Re,We,We_Re,splashing_spectrum,net_impact_type,splashing_type,volume_fraction_type
0,0,1,0,820,0.0269,0.00679,0.00312,0,0,1.219512,1,0.013301,1492.516020,1492.302356,240.108847,2,fragmentation,splashing,0.08 .. 0.10
1,0,1,0,820,0.0269,0.00679,0.00312,0,0,1.219512,1,0.013301,1492.516020,1492.302356,240.108847,2,fragmentation,splashing,0.08 .. 0.10
2,0,1,0,820,0.0269,0.00679,0.00312,0,0,1.219512,1,0.013301,1492.516020,1492.302356,240.108847,2,fragmentation,splashing,0.08 .. 0.10
3,0,1,1,820,0.0269,0.00679,0.00300,0,0,1.219512,1,0.013833,1435.111557,1434.906112,233.148786,2,fragmentation,splashing,0.08 .. 0.10
4,0,1,1,820,0.0269,0.00679,0.00300,0,0,1.219512,1,0.013833,1435.111557,1434.906112,233.148786,2,fragmentation,splashing,0.08 .. 0.10


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 372 entries, 0 to 371
Data columns (total 19 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   net_impact                       372 non-null    int64  
 1   splashing                        372 non-null    int64  
 2   wettability                      372 non-null    int64  
 3   liquid_density                   372 non-null    int64  
 4   surface_tension                  372 non-null    float64
 5   viscosity                        372 non-null    float64
 6   droplet_diameter                 372 non-null    float64
 7   inclination                      372 non-null    int64  
 8   roughness_binary                 372 non-null    int64  
 9   particle_liquid_density_ratio    372 non-null    float64
 10  volume_fraction_binary           372 non-null    int64  
 11  particle_droplet_diameter_ratio  372 non-null    float64
 12  Re                    

# Check inclination and speed

In [3]:
def extract_agg_features_upd(source_df):
    df = source_df.copy()
    
    df['norm_velocity'] = df['velocity'] * np.cos(np.deg2rad(df['inclination']))
    
    # if (df['inclination'] > 0).all():
    #     df['norm_velocity'] = df['velocity'] / df['inclination'] * 14
    # else:
    #     df['norm_velocity'] = df['velocity']
    
    Re_numerator = df['norm_velocity'] * df['droplet_diameter'] * df['liquid_density']
    
    df['Re'] = Re_numerator/df['viscosity']
    df['We'] = df['norm_velocity'] * Re_numerator / df['surface_tension']
    
    df['We_Re'] = df['We']**(1/2) * df['Re']**(1/4)
    
    return df

def extract_features_upd(source_df):
    df = source_df.copy()
    df = extract_agg_features_upd(df)
    df = ma.get_poly_df(df)
    
    df['inclination'] = 0
    
    return df

def get_contour_df_upd(
    df_model,
    net_impact_model_features:list=None,
    splashing_model_features:list=None,
    velocity:np.ndarray=np.linspace(0.0, 7.0, 50),
    particle_liquid_density_ratio:np.ndarray=np.linspace(0.3, 1.9, 50),
    particle_mean_diameter:np.ndarray=np.linspace(20e-6, 400e-6, 50),
    const_params:list=[
        'wettability',
        'inclination',
        'roughness_binary',
        'volume_fraction_binary',
        'droplet_diameter',
        'liquid_density',
        'surface_tension',
        'viscosity',
    ],
    density_params:list=[
        'particle_liquid_density_ratio'
    ],
    diameter_params:list=[
        'particle_droplet_diameter_ratio', 
    ],
    dynamic_params:list=[
        'velocity', 
        'Re', 
        'We', 
        'We_Re'
    ],
    verbose:bool=True,
):
    df_model = df_model.copy()

    vel_dens_df = ma.create_mesh_df(
        velocity=velocity,
        particle_liquid_density_ratio=particle_liquid_density_ratio
    )

    vel_diam_df = ma.create_mesh_df(
        velocity=velocity,
        particle_mean_diameter=particle_mean_diameter,
    )

    const_params_list_dens = const_params.copy()
    const_params_list_dens.extend(diameter_params)
    const_params_list_diam = const_params.copy()
    const_params_list_diam.extend(density_params)

    if verbose:
        print('vel_dens_df')
        display(vel_dens_df)
        print('vel_diam_df')
        display(vel_diam_df)
        print('const_params_list_dens')
        display(const_params_list_dens)
        print('const_params_list_diam')
        display(const_params_list_diam)

    dens_pred_df = ma.create_dataframe(
        const_params=ma.get_const_params(
            df_model, 
            const_params_list_dens
        ),
        variable_df=vel_dens_df
    )

    diam_pred_df = ma.create_dataframe(
        const_params=ma.get_const_params(
            df_model, 
            const_params_list_diam
        ),
        variable_df=vel_diam_df
    )
    diam_pred_df['particle_droplet_diameter_ratio'] = (
        diam_pred_df['particle_mean_diameter']
        / df_model['droplet_diameter'].median()
    )
    diam_pred_df = diam_pred_df.drop('particle_mean_diameter', axis=1)

    if set(diam_pred_df.columns) == set(dens_pred_df.columns):
        print('Dataframes are equal')
    else:
        print('WARNING: Dataframes are NOT equal')
    
    # TODO: Check, what if velocity was corrected?
    dens_pred_df = extract_features_upd(dens_pred_df)
    diam_pred_df = extract_features_upd(diam_pred_df)

    if verbose:
        print('dens_pred_df after We, Re extraction')
        dens_pred_df.info()
        print('diam_pred_df after We, Re extraction')
        diam_pred_df.info()

    if not((net_impact_model_features is None) or (splashing_model_features is None)):
        net_impact_columns = set(net_impact_model_features)
        extra_columns = net_impact_columns - set(dens_pred_df.columns)
        if len(extra_columns)>0:
            print('Net-impact: columns for additional creation')
            display(extra_columns)
        else:
            print('Net-impact: No columns are needed for creation')

        splashing_columns = set(splashing_model_features)
        extra_columns = splashing_columns - set(dens_pred_df.columns)
        if len(extra_columns)>0:
            print('Splashing: columns for additional creation')
            display(extra_columns)
        else:
            print('Splashing: No columns are needed for creation')
        
    print('Dataframes are prepared')
    
    return dens_pred_df, diam_pred_df

In [21]:
target_model_name = {
    'splashing': {
        'logisticregression': ['logisticregression_splashing_df_modelling_dimensionless_ordenc', [0.5]],
    },
    'net_impact': {
        'logisticregression': ['logisticregression_net_impact_df_modelling_dimensionless_ordenc', [0.5]],
    }
}

level = 0.70

y_content = [
    ('particle_liquid_density_ratio', '$\\rho_{p}/\\rho_{l}$'),
    ('particle_droplet_diameter_ratio', '$d_p/D_{drop}$')
]

models_folder = Path('..', 'results', 'models_modelling_2')

feature_names = ['wettability', 'inclination', ('particle_diameter_cat', 'particle_density')]


save_plots = False
save_path = Path('..', 'results')
save_prefix = 'submit'

fig, axes = plt.subplots(
    4, 2,
    figsize=(12, 12),
    # width_ratios=[1,1,1,1,0.05],
    dpi=600,
)

def plot_full_column(
    plot_masks,
    labels,
    colors,
    col_index,
    plot_half_column
):
    for k, (y_feature_name, y_label) in enumerate(y_content): # density or diameter
    
        plot_half_column(
            plot_masks=plot_masks,
            labels=labels,
            colors=colors,
            col_index=col_index,
            k=k,
            y_feature_name=y_feature_name,
            y_label=y_label
        )

def plot_half_column(
    plot_masks,
    labels,
    colors,
    col_index,
    k,
    y_feature_name,
    y_label
):
    for i, target in enumerate(target_model_name):
        model_name = 'logisticregression'
        
        full_model_name, levels = target_model_name[target][model_name]
        model_path = Path(models_folder, full_model_name)
        if not os.path.isfile(model_path):
            print(f"ERROR: {full_model_name}")
        model = joblib.load(model_path)
        # display(model_path)
        # display(model)
        
        if 'catboostclassifier' in model_name:
            model_features = model[0].feature_names_
        else:
            model_features = model[0].feature_names_in_
        # display(model_features)
        target_levels = levels
        
        # Predictions and plot
        model_list = [
            (target, model, model_features),
        ]
        
        ax = axes[i + 2*k, col_index]
        # ax.set_title(f'{target}, {y_label}')
    
        # Predictions and plot
        model_list = [
            (target, model, model_features),
        ]
        
        legend_elements = []
        for j, plot_mask in enumerate(plot_masks):
            df_model = df[plot_mask]
        
            pred_dfs = ma.get_contour_df(
                df_model=df_model,
                velocity=np.linspace(0.0, 6.5, 50),
                verbose=False,
            )
        
            pred_df_res = ma.predict_all_proba(pred_dfs[k], model_list)
        
            ax, contplot = ma.plot_WeRe_level(
                contour_df=pred_df_res,
                impact_type_name=target,
                y_feature_name=y_feature_name,
                y_label=y_label,
                color=colors[j],
                levels_contour=[level],
                ax=ax
            )
            legend_elements.append(contplot.legend_elements())
        
        # Get legend
        handles = []
        for legend_element in legend_elements:
            handles.append(legend_element[0][0])
        # for ax in axes.flat:
        ax.legend(handles=handles, labels=labels)
        
        # if y_feature_name == 'particle_droplet_diameter_ratio':
        #     limits = [0.01, 0.12]
        #     ax.set_yticks(np.arange(*limits, 0.03))
        #     ax.set_ylim(limits)
        # else:
        #     limits = [0.3, 1.8]
        #     ax.set_yticks(np.arange(*limits, 0.3))
        #     ax.set_ylim(limits)
        
        if y_feature_name == 'particle_droplet_diameter_ratio':
            limits = [-0.01, 0.15]
            ax.set_yticks(np.arange(*limits, 0.02))
            ax.set_ylim(limits)
        else:
            limits = [-0.2, 2.0]
            ax.set_yticks(np.arange(*limits, 0.2))
            ax.set_ylim(limits)
        
        # limits = [-100, 300]
        # ax.set_xticks(np.arange(*limits, 50))
        # ax.set_xlim(limits)
        
        limits = [170, 220]
        ax.set_xticks(np.arange(*limits, 5))
        ax.set_xlim(limits)
        
        # limits = [0, 300]
        # ax.set_xticks(np.arange(*limits, 50))
        # ax.set_xlim(limits)
            
        # if i + 2*k != 3:
        #     ax.xaxis.set_ticklabels([])
        #     ax.set_xlabel('')
            
        # if col_index != 0:
        #     ax.yaxis.set_ticklabels([])
        #     ax.set_ylabel('')


def plot_new_half_column(
    plot_masks,
    labels,
    colors,
    col_index,
    k,
    y_feature_name,
    y_label
):
    for i, target in enumerate(target_model_name):
        model_name = 'logisticregression'
        
        full_model_name, levels = target_model_name[target][model_name]
        model_path = Path(models_folder, full_model_name)
        if not os.path.isfile(model_path):
            print(f"ERROR: {full_model_name}")
        model = joblib.load(model_path)
        # display(model_path)
        # display(model)
        
        if 'catboostclassifier' in model_name:
            model_features = model[0].feature_names_
        else:
            model_features = model[0].feature_names_in_
        # display(model_features)
        target_levels = levels
        
        # Predictions and plot
        model_list = [
            (target, model, model_features),
        ]
        
        ax = axes[i + 2*k, col_index]
        # ax.set_title(f'{target}, {y_label}')
    
        # Predictions and plot
        model_list = [
            (target, model, model_features),
        ]
        
        legend_elements = []
        for j, plot_mask in enumerate(plot_masks):
            df_model = df[plot_mask]
        
            pred_dfs = get_contour_df_upd(
                df_model=df_model,
                velocity=np.linspace(0.0, 6.5, 50),
                verbose=False,
            )
        
            pred_df_res = ma.predict_all_proba(pred_dfs[k], model_list)
        
            ax, contplot = ma.plot_WeRe_level(
                contour_df=pred_df_res,
                impact_type_name=target,
                y_feature_name=y_feature_name,
                y_label=y_label,
                color=colors[j],
                ax=ax,
                levels_contour=[level],
            )
            legend_elements.append(contplot.legend_elements())
        
        # Get legend
        handles = []
        for legend_element in legend_elements:
            handles.append(legend_element[0][0])
        # for ax in axes.flat:
        ax.legend(handles=handles, labels=labels)
        
        # if y_feature_name == 'particle_droplet_diameter_ratio':
        #     limits = [0.01, 0.12]
        #     ax.set_yticks(np.arange(*limits, 0.03))
        #     ax.set_ylim(limits)
        # else:
        #     limits = [0.3, 1.8]
        #     ax.set_yticks(np.arange(*limits, 0.3))
        #     ax.set_ylim(limits)
        
        if y_feature_name == 'particle_droplet_diameter_ratio':
            limits = [-0.01, 0.15]
            ax.set_yticks(np.arange(*limits, 0.02))
            ax.set_ylim(limits)
        else:
            limits = [-0.2, 2.0]
            ax.set_yticks(np.arange(*limits, 0.2))
            ax.set_ylim(limits)
        
        limits = [170, 220]
        ax.set_xticks(np.arange(*limits, 5))
        ax.set_xlim(limits)
        
        # limits = [0, 300]
        # ax.set_xticks(np.arange(*limits, 50))
        # ax.set_xlim(limits)
            
        # if i + 2*k != 3:
        #     ax.xaxis.set_ticklabels([])
        #     ax.set_xlabel('')
            
        # if col_index != 0:
        #     ax.yaxis.set_ticklabels([])
        #     ax.set_ylabel('')


# INCLINATION: Let us set lyophilic smooth substrate with different inclinations
col_index = 0
wet_mask = df['wettability'] == 0 # 'neutral'
roughness_mask = df['roughness_binary'] == 0
# inclination_mask = df['inclination'] == 0

# Wettability mask lists
plot_masks = [
    df['inclination'] == 0,
    df['inclination'] == 20,
    df['inclination'] == 45,
]
for i, plot_mask in enumerate(plot_masks):
    plot_masks[i] = (
        plot_mask & roughness_mask & wet_mask
    )

labels = [
    '0º',
    '20º',
    '45º'
]

colors = [
    'blue',
    'green',
    'red',
]

plot_full_column(
    plot_masks=plot_masks,
    labels=labels,
    colors=colors,
    col_index=col_index,
    plot_half_column=plot_half_column,
)

# INCLINATION: Plot with new approach
col_index = 1

plot_full_column(
    plot_masks=plot_masks,
    labels=labels,
    colors=colors,
    col_index=col_index,
    plot_half_column=plot_new_half_column,
)

letters = 'abcdefgh'

for i, ax in enumerate(axes.flatten()):
    ax.set_title(letters[i])


fig.set_layout_engine(layout='compressed')


if save_plots:
    plt.savefig(
        Path(
            save_path, 
            f'{save_prefix}_WeRe_comparison.pdf',
        ),
        dpi=600
    )

Dataframes are equal
Dataframes are prepared
Dataframes are equal
Dataframes are prepared
Dataframes are equal
Dataframes are prepared
Dataframes are equal
Dataframes are prepared
Dataframes are equal
Dataframes are prepared
Dataframes are equal
Dataframes are prepared
Dataframes are equal
Dataframes are prepared
Dataframes are equal
Dataframes are prepared
Dataframes are equal
Dataframes are prepared
Dataframes are equal
Dataframes are prepared
Dataframes are equal
Dataframes are prepared
Dataframes are equal
Dataframes are prepared
Dataframes are equal
Dataframes are prepared
Dataframes are equal
Dataframes are prepared
Dataframes are equal
Dataframes are prepared
Dataframes are equal
Dataframes are prepared
Dataframes are equal
Dataframes are prepared
Dataframes are equal
Dataframes are prepared
Dataframes are equal
Dataframes are prepared
Dataframes are equal
Dataframes are prepared
Dataframes are equal
Dataframes are prepared
Dataframes are equal
Dataframes are prepared
Dataframes

In [5]:
df

,net_impact,splashing,wettability,liquid_density,surface_tension,viscosity,droplet_diameter,inclination,roughness_binary,particle_liquid_density_ratio,volume_fraction_binary,particle_droplet_diameter_ratio,Re,We,We_Re,splashing_spectrum,net_impact_type,splashing_type,volume_fraction_type
0,0,1,0,820,0.0269,0.00679,0.00312,0,0,1.219512,1,0.013301,1492.516020,1492.302356,240.108847,2,fragmentation,splashing,0.08 .. 0.10
1,0,1,0,820,0.0269,0.00679,0.00312,0,0,1.219512,1,0.013301,1492.516020,1492.302356,240.108847,2,fragmentation,splashing,0.08 .. 0.10
2,0,1,0,820,0.0269,0.00679,0.00312,0,0,1.219512,1,0.013301,1492.516020,1492.302356,240.108847,2,fragmentation,splashing,0.08 .. 0.10
3,0,1,1,820,0.0269,0.00679,0.00300,0,0,1.219512,1,0.013833,1435.111557,1434.906112,233.148786,2,fragmentation,splashing,0.08 .. 0.10
4,0,1,1,820,0.0269,0.00679,0.00300,0,0,1.219512,1,0.013833,1435.111557,1434.906112,233.148786,2,fragmentation,splashing,0.08 .. 0.10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
367,0,0,2,1180,0.0679,0.02310,0.00334,20,0,1.016949,1,0.082335,337.914500,227.687593,64.695100,0,fragmentation,no splashing,0.08 .. 0.10
368,0,1,0,1180,0.0679,0.02310,0.00330,20,0,1.016949,1,0.083333,333.867620,224.960796,64.113132,1,fragmentation,semi splashing,0.08 .. 0.10
369,0,1,0,1180,0.0679,0.02310,0.00327,20,0,1.016949,1,0.084098,330.832459,222.915698,63.675499,1,fragmentation,semi splashing,0.08 .. 0.10
370,0,0,1,1180,0.0679,0.02310,0.00335,20,0,1.016949,1,0.082090,338.926220,228.369293,64.840319,0,fragmentation,no splashing,0.08 .. 0.10


In [6]:

def update_data(df):
    df['velocity'] = df['velocity'] * np.cos(np.deg2rad(df['inclination']))

    Re_numerator = df['velocity'] * df['droplet_diameter'] * df['liquid_density']

    df['Re'] = Re_numerator/df['viscosity']
    df['We'] = df['velocity'] * Re_numerator / df['surface_tension']

    df['We_Re'] = df['We']**(1/2) * df['Re']**(1/4)
    
    return df
    
df_main = update_data(df_main)

In [7]:
np.abs((df['Re'] - df_main['Re'])/df['Re']).max()

0.2928932188134526

In [8]:
np.abs((df['We'] - df_main['We'])/df['We']).max()

0.5000000000000002

In [9]:
np.abs((df['We_Re'] - df_main['We_Re'])/df['We_Re']).max()

0.35158022267449557